# Clase 1.1: Preprocesamiento de texto

En las diapositivas nos preguntamos qué ve un computador cuando lee: una cadena de caracteres. En este notebook convertimos noticias reales en listas de unidades que se pueden contar. Normalizamos el texto, lo cortamos en tokens de tres maneras y reducimos cada palabra a su forma base. En el notebook 1.2 contamos esas unidades.

**Objetivos de la clase**

- Normalizar un texto y ver qué información se pierde en el camino.
- Tokenizar separando por espacios, con una expresión regular y con las reglas de spaCy.
- Entender cómo BPE y WordPiece aprenden un vocabulario de subpalabras a partir de los datos, y qué cambia con el tamaño de ese vocabulario.
- Distinguir lematización de stemming y saber cuándo conviene cada una.

## 0. Preparación

Instalamos lo que Colab no trae: `tokenizers` (subpalabras), `nltk` (expresiones regulares y stemming) y el modelo de spaCy para español.

In [1]:
# Solo instalamos lo que Colab no trae. No actualizamos numpy ni pandas
# con -U: eso rompe el entorno de Colab.
!pip install -q tokenizers nltk
!python -m spacy download es_core_news_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')


## 1. Los datos: noticias de BBC Mundo

Trabajamos con la parte en español de XL-Sum (Hasan et al., 2021), un conjunto de noticias de BBC Mundo publicado en el Hub de Hugging Face. Cada noticia trae su texto (`text`), su título (`title`), un resumen (`summary`) y la dirección original (`url`). Cada noticia es un **documento**, y el conjunto de documentos es el **corpus** (en plural, *corpora*). Cargamos las primeras 5.000.

In [2]:
import datasets

# Leemos la copia en parquet de XL-Sum, que no depende de la versión
# de `datasets`. Nos quedamos con las primeras 5.000 noticias.
base = "hf://datasets/csebuetnlp/xlsum@refs/convert/parquet/spanish"
spanish_news = {
    "train": datasets.load_dataset(
        "parquet",
        data_files={"train": f"{base}/train/*.parquet"},
        split="train[:5000]",
    )
}
spanish_news["train"][0]

{'id': '140930_ultnot_siria_onu_comida_ch',
 'url': 'https://www.bbc.com/mundo/ultimas_noticias/2014/09/140930_ultnot_siria_onu_comida_ch',
 'title': 'ONU dice que peligran raciones de comida para Siria',
 'summary': 'La directora de ayuda humanitaria de Naciones Unidas, Valerie Amos, advirtió que de no invertir más dinero, el Programa Mundial de Alimentos tendrá que detener sus operaciones en Siria en dos meses.',
 'text': 'De no recibir más dinero, las raciones de la ONU en Siria se terminarán en dos meses. En un informe al Consejo de Seguridad de la ONU, Amos dijo que las raciones del PMA destinadas a los 4.000.000 de sirios ya han sido recortadas para poder llegar a la mayor cantidad de personas posible. Amos también hizo un llamado a juntar suministros para proteger a los sirios del frío, en miras al próximo invierno.'}

## 2. ¿Cuándo dos textos dicen lo mismo?

Para nosotros, «Estados Unidos», «estados unidos» y «ESTADOS UNIDOS» son lo mismo. Para Python son cadenas distintas:

In [3]:
"Estados Unidos" == "estados unidos"

False

Si contamos palabras así, cada variante cuenta por separado. Llevar las variantes a una sola forma canónica se llama **normalización** (normalization). Lo más común es pasar a minúsculas, quitar las tildes y quitar todo lo que no sea letra. Escribamos una función que hace las tres cosas.

In [4]:
import re

def normalize(text: str, remove_tildes: bool = True) -> str:
    # Todo a minúsculas
    text = text.lower()
    # Todo lo que no sea una letra pasa a ser un espacio
    text = re.sub(r"[^a-zñüáéíóú]", " ", text)
    if remove_tildes:
        for con_tilde, sin_tilde in zip("áéíóú", "aeiou"):
            text = text.replace(con_tilde, sin_tilde)
    return text

La aplicamos a todo el corpus con `Dataset.map()`, que le agrega a cada noticia un campo nuevo, `normalized_text`.

In [5]:
spanish_news_normalized = spanish_news["train"].map(
    lambda noticia: {"normalized_text": normalize(noticia["text"])}
)
spanish_news_normalized

Dataset({
    features: ['id', 'url', 'title', 'summary', 'text', 'normalized_text'],
    num_rows: 5000
})

Comparemos un texto antes y después de normalizar, con y sin tildes.

In [6]:
ejemplo = spanish_news["train"][10]["text"][:300]

print("Original:")
print(ejemplo)
print()
print("Normalizado, sin tildes:")
print(normalize(ejemplo, remove_tildes=True))
print()
print("Normalizado, con tildes:")
print(normalize(ejemplo, remove_tildes=False))

Original:
Un alto funcionario dijo a la BBC que no se consultó a Estados Unidos sobre los ataques y que fue sorprendido con la guardia baja. Según informes, las incursiones en Trípoli fueron realizadas por fuerzas de Emiratos Árabes Unidos usando bases en Egipto. Las autoridades egipcias negaron su participac

Normalizado, sin tildes:
un alto funcionario dijo a la bbc que no se consulto a estados unidos sobre los ataques y que fue sorprendido con la guardia baja  segun informes  las incursiones en tripoli fueron realizadas por fuerzas de emiratos arabes unidos usando bases en egipto  las autoridades egipcias negaron su participac

Normalizado, con tildes:
un alto funcionario dijo a la bbc que no se consultó a estados unidos sobre los ataques y que fue sorprendido con la guardia baja  según informes  las incursiones en trípoli fueron realizadas por fuerzas de emiratos árabes unidos usando bases en egipto  las autoridades egipcias negaron su participac


Noten lo que se perdió. «BBC» ya no se distingue de una palabra común. Sin tildes, «consultó» pasa a «consulto», que es otra forma del verbo, y «Trípoli» pasa a «tripoli». Para contar palabras eso ayuda, pero para reconocer una sigla o un nombre propio estorba. No hay una normalización correcta para todo: depende de la tarea.

## 3. ¿Dónde termina una palabra?

Antes de contar palabras hay que cortar el texto en pedazos. Cada pedazo que resulta se llama **token**, y cortar el texto en tokens es **tokenizar**. Los tokens distintos son los **tipos** (types), y el conjunto de tipos es el **vocabulario** del corpus.

Probemos lo más simple, separar por espacios. Sobre el texto normalizado funciona bien:

In [7]:
tokens = spanish_news_normalized[0]["normalized_text"].split()
print(" | ".join(tokens[:20]))

de | no | recibir | mas | dinero | las | raciones | de | la | onu | en | siria | se | terminaran | en | dos | meses | en | un | informe


¿Y sobre el texto original?

In [8]:
tokens = spanish_news_normalized[0]["text"].split()
print(" | ".join(tokens[:20]))

De | no | recibir | más | dinero, | las | raciones | de | la | ONU | en | Siria | se | terminarán | en | dos | meses. | En | un | informe


Noten «dinero,» y «meses.»: la puntuación queda pegada a la palabra. Si contamos tipos, «meses» y «meses.» serían dos palabras distintas. Necesitamos reglas para separar los signos.

### Reglas con expresiones regulares

Una forma de escribir esas reglas es con una expresión regular (regular expression): un patrón por cada cosa que queremos que cuente como un token. Probemos con una frase que tiene una sigla, un monto y un porcentaje.

In [9]:
import nltk

texto_ejemplo = "El cartel de EE.UU. cuesta $12.40... ¡un 25% de descuento!"

# Las alternativas más específicas van primero: si \w+ fuera antes,
# separaría "$12.40" en "12", "." y "40".
# Los grupos son no capturantes, (?:...), para que regexp_tokenize
# devuelva el token completo y no solo lo que está dentro del grupo.
patron = r'''(?x)            # modo verbose: permite comentarios
    (?:[A-ZÁÉÍÓÚ]+\.)+       # siglas con puntos, como EE.UU.
  | \$?\d+(?:\.\d+)?%?       # montos y porcentajes: $12.40, 25%
  | \w+(?:-\w+)*             # palabras, con guiones internos
  | \.\.\.                   # puntos suspensivos
  | [¡¿!.,;"'?():_-]         # signos sueltos
'''

nltk.regexp_tokenize(texto_ejemplo, patron)

['El',
 'cartel',
 'de',
 'EE.UU.',
 'cuesta',
 '$12.40',
 '...',
 '¡',
 'un',
 '25%',
 'de',
 'descuento',
 '!']

Cada patrón resolvió un caso. El costo es que hay que anticiparlos todos: una sigla sin puntos o un precio con coma decimal necesitan reglas nuevas.

### ¿Normalizar antes o después?

¿Qué pasa si normalizamos antes de tokenizar? Comparemos las dos rutas sobre la misma frase.

In [10]:
frase = "El cartel de EE.UU. llegó a Chile"

# Ruta A: normalizar primero y después separar por espacios
ruta_a = normalize(frase).split()

# Ruta B: tokenizar con reglas primero y después pasar a minúsculas
ruta_b = [token.lower() for token in nltk.regexp_tokenize(frase, patron)]

print("A (normalizar y tokenizar):", ruta_a)
print("B (tokenizar y normalizar):", ruta_b)

A (normalizar y tokenizar): ['el', 'cartel', 'de', 'ee', 'uu', 'llego', 'a', 'chile']
B (tokenizar y normalizar): ['el', 'cartel', 'de', 'ee.uu.', 'llegó', 'a', 'chile']


Al normalizar primero, «EE.UU.» se rompe en «ee» y «uu», porque `normalize` borró los puntos que la regla necesitaba. Por eso se tokeniza primero y se normaliza cada token después.

### El tokenizador de spaCy

[spaCy](https://spacy.io) es una biblioteca de NLP que ofrece, para cada idioma, un *pipeline* ya entrenado que tokeniza, lematiza, etiqueta la categoría gramatical de cada palabra y reconoce entidades. El que instalamos, `es_core_news_sm`, es el pipeline pequeño (*sm*) para español (*es*), entrenado con noticias (*news*).

Por ahora usamos solo su tokenizador, que no se entrena con datos: aplica reglas del español escritas a mano. Primero separa por espacios y después revisa cada fragmento:

| Regla | Qué revisa | Ejemplos |
|---|---|---|
| Excepción (`SPECIAL`) | ¿Es una abreviación conocida? Se deja entera. | `Sr.`, `EE.UU.` |
| Prefijo (`PREFIX`) | ¿Empieza con un signo que hay que separar? | `¿`, `¡`, `(` |
| Sufijo (`SUFFIX`) | ¿Termina con un signo que hay que separar? | `,` `.` `?` `!` `)` |
| Infijo (`INFIX`) | ¿Tiene en el medio un signo que separa dos tokens? | `perro/gato`, `hola...chao` |
| Token (`TOKEN`) | Lo que queda es un token. | `Pérez`, `10:30`, `ex-presidente` |

In [11]:
import spacy

spacy_tokenizer = spacy.load("es_core_news_sm").tokenizer

In [12]:
# explain() muestra qué regla produjo cada token
frase = "¿El Sr. Pérez, ex-presidente de EE.UU., llegó a las 10:30 a.m.?"
for regla, token in spacy_tokenizer.explain(frase):
    print(f"{token:<15} <- {regla}")

¿               <- PREFIX
El              <- TOKEN
Sr.             <- SPECIAL-1
Pérez           <- TOKEN
,               <- SUFFIX
ex-presidente   <- TOKEN
de              <- TOKEN
EE.UU.          <- SPECIAL-1
,               <- SUFFIX
llegó           <- TOKEN
a               <- TOKEN
las             <- TOKEN
10:30           <- TOKEN
a.m             <- TOKEN
.               <- SUFFIX
?               <- SUFFIX


- `¿` se separa como prefijo, y las comas y el `?` final como sufijos.
- `Sr.` y `EE.UU.` son excepciones: el punto es parte de la abreviación.
- `ex-presidente` y `10:30` quedan enteros.
- `a.m.` no está entre las excepciones del español y queda como `a.m` y `.`. Ninguna lista de reglas cubre todos los casos.

Tokenicemos la primera noticia.

In [13]:
noticia = spacy_tokenizer(spanish_news["train"][0]["text"])
tokens = [token.text for token in noticia]
print(" | ".join(tokens[:45]))

De | no | recibir | más | dinero | , | las | raciones | de | la | ONU | en | Siria | se | terminarán | en | dos | meses | . | En | un | informe | al | Consejo | de | Seguridad | de | la | ONU | , | Amos | dijo | que | las | raciones | del | PMA | destinadas | a | los | 4.000.000 | de | sirios | ya | han


Ahora «dinero», «meses» y sus signos son tokens separados, y «4.000.000» queda entero.

## 4. ¿Qué hacemos con una palabra que nunca vimos?

Supongamos que armamos el vocabulario con los tipos de estas 5.000 noticias y después llega una noticia con «desinfodemia». Si no apareció antes, es una palabra fuera de vocabulario (out of vocabulary, OOV), y un modelo no tiene cómo representarla.

Los tokenizadores modernos cortan en **subpalabras**, piezas que sí aparecieron antes: «desinfodemia» puede armarse con «des», «info» y «demia». Las piezas se aprenden del corpus. Las palabras frecuentes quedan enteras y las raras se arman con varias piezas.

### BPE

El algoritmo más simple es **BPE** (Byte-Pair Encoding). Nació en 1994 como un método de compresión (Gage), y Sennrich, Haddow y Birch lo adaptaron en 2016 para la traducción automática. Funciona así:

1. El vocabulario empieza con todos los caracteres del corpus.
2. Se busca el par de símbolos vecinos más frecuente.
3. Ese par se fusiona en un símbolo nuevo, que entra al vocabulario.
4. Se repite hasta llegar al tamaño de vocabulario que elegimos.

Los pares se cuentan dentro de cada palabra, así que una pieza nunca cruza de una palabra a otra. Los modelos GPT usan BPE: lo que lee ChatGPT son piezas de este tipo, no palabras.

Probemos con seis formas verbales. ¿Qué pieza creen que aparecerá primero?

In [14]:
import collections

def bpe_pasos(palabras: list[str], n_fusiones: int) -> None:
    # Cada palabra empieza como una lista de caracteres
    segmentadas = [list(palabra) for palabra in palabras]
    print("Inicio:", [" ".join(simbolos) for simbolos in segmentadas])

    for paso in range(1, n_fusiones + 1):
        # Contamos los pares de símbolos vecinos dentro de cada palabra
        pares = collections.Counter()
        for simbolos in segmentadas:
            pares.update(zip(simbolos, simbolos[1:]))
        par, frecuencia = pares.most_common(1)[0]
        nuevo = "".join(par)

        # Reemplazamos cada aparición del par por el símbolo nuevo
        for k, simbolos in enumerate(segmentadas):
            fusionados = []
            i = 0
            while i < len(simbolos):
                if tuple(simbolos[i:i + 2]) == par:
                    fusionados.append(nuevo)
                    i += 2
                else:
                    fusionados.append(simbolos[i])
                    i += 1
            segmentadas[k] = fusionados

        print(f"Paso {paso}: {par} aparece {frecuencia} veces -> {nuevo}")

    print("Final:", [" ".join(simbolos) for simbolos in segmentadas])


formas = ["cantando", "bailando", "saltando", "cantaba", "bailaba", "saltaba"]
bpe_pasos(formas, n_fusiones=8)

Inicio: ['c a n t a n d o', 'b a i l a n d o', 's a l t a n d o', 'c a n t a b a', 'b a i l a b a', 's a l t a b a']
Paso 1: ('a', 'n') aparece 5 veces -> an
Paso 2: ('b', 'a') aparece 5 veces -> ba
Paso 3: ('an', 'd') aparece 3 veces -> and
Paso 4: ('and', 'o') aparece 3 veces -> ando
Paso 5: ('a', 'ba') aparece 3 veces -> aba
Paso 6: ('c', 'an') aparece 2 veces -> can
Paso 7: ('can', 't') aparece 2 veces -> cant
Paso 8: ('ba', 'i') aparece 2 veces -> bai
Final: ['cant ando', 'bai l ando', 's a l t ando', 'cant aba', 'bai l aba', 's a l t aba']


En ocho pasos aparecieron `ando` y `aba`, las terminaciones del gerundio y del imperfecto, sin que nadie le enseñara gramática al algoritmo: solo son las secuencias que más se repiten. Las piezas no siempre coinciden con la morfología, como muestra `bai`, que salió de juntar `ba` con la `i` de «bailando».

### Un tokenizador entrenado con las noticias

La biblioteca `tokenizers` de Hugging Face implementa estos algoritmos. Entrenemos sobre nuestras noticias un tokenizador **WordPiece**, el que usa BERT (clase 5). WordPiece también parte de los caracteres y fusiona pares. Las piezas que continúan una palabra llevan el prefijo `##`.

In [15]:
import tokenizers

tokenizer = tokenizers.Tokenizer(tokenizers.models.WordPiece())

Le agregamos los mismos pasos de `normalize()`: minúsculas, sin tildes y solo letras.

In [16]:
normalizer = tokenizers.normalizers.Sequence([
    tokenizers.normalizers.Lowercase(),
    # NFD separa cada letra de su tilde (á -> a + ´), y StripAccents
    # borra la tilde
    tokenizers.normalizers.NFD(),
    tokenizers.normalizers.StripAccents(),
    # Lo que no sea letra o espacio pasa a ser un espacio
    tokenizers.normalizers.Replace(tokenizers.Regex(r"[^a-z ]"), " "),
])

normalizer.normalize_str(spanish_news["train"][0]["text"])

'de no recibir mas dinero  las raciones de la onu en siria se terminaran en dos meses  en un informe al consejo de seguridad de la onu  amos dijo que las raciones del pma destinadas a los           de sirios ya han sido recortadas para poder llegar a la mayor cantidad de personas posible  amos tambien hizo un llamado a juntar suministros para proteger a los sirios del frio  en miras al proximo invierno '

Lo conectamos al tokenizador. Antes de buscar subpalabras separamos por espacios, para que ninguna pieza cruce de una palabra a otra. Después entrenamos con el tamaño de vocabulario por defecto, 30.000 piezas.

In [17]:
tokenizer.normalizer = normalizer
tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()

trainer = tokenizers.trainers.WordPieceTrainer()
tokenizer.train_from_iterator(spanish_news["train"]["text"], trainer)
tokenizer.get_vocab_size()

30000

`encode()` devuelve, entre otras cosas, dos listas: `tokens`, el texto de cada pieza, e `ids`, el número de cada pieza en el vocabulario. Un modelo solo ve los `ids`.

In [18]:
tokenized_output = tokenizer.encode(spanish_news["train"][0]["text"])
print(" | ".join(tokenized_output.tokens[:45]))
print(tokenized_output.ids[:45])

de | no | recibir | mas | dinero | las | raciones | de | la | onu | en | siria | se | terminaran | en | dos | meses | en | un | informe | al | consejo | de | seguridad | de | la | onu | amos | dijo | que | las | raciones | del | pm | ##a | destinadas | a | los | de | sirios | ya | han | sido | recor | ##tadas
[52, 103, 872, 141, 1162, 104, 27684, 52, 65, 2767, 66, 2456, 83, 24437, 66, 393, 1041, 66, 71, 1992, 100, 3058, 52, 988, 52, 65, 2767, 24126, 443, 67, 104, 27684, 107, 14027, 31, 16914, 0, 84, 52, 9533, 432, 356, 505, 2048, 2277]


Casi todas las palabras quedaron enteras, porque aparecen muchas veces en las 5.000 noticias. Las menos frecuentes se cortan: la sigla «PMA» queda como `pm` y `##a`, y «recortadas» como `recor` y `##tadas`. «4.000.000» desapareció, porque el normalizador borra todo lo que no sea letra.

### ¿Cuántas piezas necesita el vocabulario?

El tamaño del vocabulario (`vocab_size`) lo elegimos nosotros. ¿Cómo se verá el título de la primera noticia si el vocabulario tiene solo 50 piezas? ¿Y con 5.000?

In [19]:
titulo = spanish_news["train"][0]["title"]
print(titulo)
print()

for vocab_size in [50, 100, 500, 1000, 5000]:
    tokenizer_prueba = tokenizers.Tokenizer(tokenizers.models.WordPiece())
    tokenizer_prueba.normalizer = normalizer
    tokenizer_prueba.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
    trainer_prueba = tokenizers.trainers.WordPieceTrainer(
        vocab_size=vocab_size
    )
    tokenizer_prueba.train_from_iterator(
        spanish_news["train"]["text"], trainer_prueba
    )
    salida = tokenizer_prueba.encode(titulo)
    print(f"vocab_size = {vocab_size:>5}: {len(salida.tokens):>2} tokens")
    print("   ", " | ".join(salida.tokens))

ONU dice que peligran raciones de comida para Siria






vocab_size =    50: 43 tokens
    o | ##n | ##u | d | ##i | ##c | ##e | q | ##u | ##e | p | ##e | ##l | ##i | ##g | ##r | ##a | ##n | r | ##a | ##c | ##i | ##o | ##n | ##e | ##s | d | ##e | c | ##o | ##m | ##i | ##d | ##a | p | ##a | ##r | ##a | s | ##i | ##r | ##i | ##a





vocab_size =   100: 29 tokens
    o | ##n | ##u | d | ##ic | ##e | que | p | ##e | ##l | ##i | ##g | ##ra | ##n | r | ##a | ##cion | ##es | de | c | ##om | ##id | ##a | par | ##a | s | ##i | ##r | ##ia



vocab_size =   500: 18 tokens
    o | ##n | ##u | dice | que | pe | ##l | ##ig | ##ran | r | ##aciones | de | com | ##ida | para | si | ##ri | ##a



vocab_size =  1000: 16 tokens
    o | ##n | ##u | dice | que | pel | ##ig | ##ran | ra | ##ciones | de | com | ##ida | para | si | ##ria



vocab_size =  5000: 11 tokens
    onu | dice | que | pelig | ##ran | ra | ##ciones | de | comida | para | siria


Con 50 piezas el vocabulario casi no tiene más que letras, y el título se corta letra por letra: 43 tokens. Con 500 aparecen palabras frecuentes enteras («dice», «que», «para») y terminaciones como `##aciones`. Con 5.000 el título queda en 11 tokens, y solo «peligran» y «raciones» siguen en dos piezas. Un vocabulario más grande deja más palabras enteras y produce secuencias más cortas, pero tiene más piezas que el modelo debe aprender.

Como antes, aplicamos el tokenizador a todo el corpus con `map()`.

In [20]:
spanish_news_tokenized = spanish_news["train"].map(
    lambda noticia: {
        "tokenized_text": tokenizer.encode(noticia["text"]).tokens
    }
)
print(" | ".join(spanish_news_tokenized[1]["tokenized_text"][:30]))

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

manifestantes | hindu | ##es | gritan | consignas | contra | el | gobierno | en | la | region | de | cachemira | administrada | por | india | las | manifestaciones | tuvieron | lugar | despues | de | los | rezo | ##s | especiales | ei | ##d | en | la


## 5. ¿«corrimos» y «correrán» son la misma palabra?

Un verbo regular del español tiene decenas de formas: «canto», «cantaste», «cantábamos», «cantarían»… Al contar palabras, cada forma es un tipo distinto, aunque todas hablen de cantar. Hay dos maneras de juntarlas.

La **lematización** (lemmatization) lleva cada palabra a su lema, la forma que aparece en el diccionario: los verbos al infinitivo, los sustantivos y adjetivos al singular. Para eso necesita saber qué papel cumple la palabra en la oración: «estrellado» viene del verbo «estrellar» en «el auto se ha estrellado», y es el adjetivo «estrellado» en «un cielo estrellado».

El **stemming** corta la terminación con reglas fijas, sin mirar el contexto, y deja una raíz (stem) que muchas veces no es una palabra: «corriendo» queda en «corr». El algoritmo clásico es el de Porter (1980), para el inglés. Aquí usamos Snowball, su sucesor, que trae reglas para el español.

| | Lematización | Stemming |
|---|---|---|
| Resultado | una palabra (`correr`) | una raíz, que a veces no es palabra (`corr`) |
| Mira el contexto | sí | no |
| Necesita | un modelo del idioma | solo reglas |
| Velocidad | más lenta | muy rápida |

Lematizamos con spaCy, ahora con el pipeline completo, y hacemos stemming con NLTK. Los dos trabajan sobre palabras enteras, así que usamos el tokenizador de spaCy y no las subpalabras de WordPiece.

In [21]:
nlp = spacy.load("es_core_news_sm")
stemmer = nltk.stem.SnowballStemmer("spanish")

Probemos con varias formas de las mismas palabras. ¿Dónde creen que se equivocará cada método?

In [22]:
palabras_prueba = [
    "corriendo", "corrimos", "correrán",
    "estudiando", "estudié", "estudiantes",
    "felices", "felizmente",
]

for palabra in palabras_prueba:
    lema = nlp(palabra)[0].lemma_
    stem = stemmer.stem(palabra)
    print(f"{palabra:15s} lema: {lema:15s} stem: {stem}")

corriendo       lema: correr          stem: corr
corrimos        lema: correr          stem: corr
correrán        lema: correr          stem: corr
estudiando      lema: estudiar        stem: estudi
estudié         lema: estudié         stem: estudi
estudiantes     lema: estudiante      stem: estudi
felices         lema: feliz           stem: felic
felizmente      lema: felizmente      stem: feliz


Noten dos errores distintos. El lematizador no reconoció «estudié» y lo dejó igual. El stemmer llevó «estudiando» y «estudiantes» a la misma raíz, `estudi`, aunque una es una acción y la otra son personas.

Ahora sobre una noticia, con la categoría gramatical (POS, part of speech) que spaCy le asigna a cada token:

In [23]:
# Usamos el texto original, con tildes y mayúsculas: el lematizador
# necesita esa información
texto = spanish_news["train"][5]["text"]
for token in list(nlp(texto))[:25]:
    print(
        f"{token.text:15s} POS: {token.pos_:6s} "
        f"lema: {token.lemma_:15s} stem: {stemmer.stem(token.text)}"
    )

A               POS: ADP    lema: a               stem: a
nadie           POS: PRON   lema: nadie           stem: nadi
le              POS: PRON   lema: él              stem: le
gusta           POS: VERB   lema: gustar          stem: gust
pensar          POS: VERB   lema: pensar          stem: pens
en              POS: ADP    lema: en              stem: en
eso             POS: PRON   lema: ese             stem: eso
,               POS: PUNCT  lema: ,               stem: ,
pero            POS: CCONJ  lema: pero            stem: per
el              POS: DET    lema: el              stem: el
manejo          POS: NOUN   lema: manejo          stem: manej
de              POS: ADP    lema: de              stem: de
la              POS: DET    lema: el              stem: la
huella          POS: NOUN   lema: huella          stem: huell
digital         POS: ADJ    lema: digital         stem: digital
"               POS: PUNCT  lema: "               stem: "
póstuma         POS: PROPN  lema: póstum

El lema de «la» es «el», y el de «le» y «se» es «él»: spaCy lleva artículos y pronombres a una forma base. El stemmer deja «pero» en `per` y «está» en `esta`, que es otra palabra.

## 6. ¿Cuándo conviene reducir el vocabulario?

Juntar variantes sirve cuando contamos palabras: una nube de palabras con «correr» se lee mejor que una con «corriendo», «corrí» y «correremos». Por eso la lematización y el stemming se usan en búsqueda, en conteos, en TF-IDF (notebook 1.2) y cuando una persona va a leer el resultado.

Con BERT o GPT casi nunca se lematiza: sus subpalabras no dejan palabras fuera de vocabulario, y el modelo aprende por su cuenta que «corriendo» y «corrí» se parecen.

## 7. Errores comunes

- **Preprocesar distinto al entrenar y al usar el modelo.** Si el modelo se entrenó con texto en minúsculas y sin tildes, todo texto nuevo tiene que pasar por el mismo proceso. Si no, aparecen tipos que el modelo nunca vio.
- **Quitar palabras muy frecuentes sin mirar la tarea.** A veces se descartan palabras como «no», «me» o «de» (las *stopwords*, que vemos en el notebook 1.2). Si a «No me gusta» le quitamos «no» y «me», queda «gusta», con el sentido al revés.
- **Mezclar codificaciones.** Un texto en UTF-8 leído como Latin-1 muestra `Ã±` en vez de `ñ`, y esos tipos basura entran al vocabulario.
- **Confiar en el stemmer sin revisar.** Puede juntar palabras que no tienen que ver, como vimos con «estudiando» y «estudiantes».

## Resumen

- La normalización junta variantes de una misma palabra, pero borra información que a veces importa, como las siglas o las tildes que distinguen dos palabras.
- Separar por espacios deja la puntuación pegada. Las reglas (expresiones regulares, spaCy) la separan, pero hay que anticipar cada caso. Primero se tokeniza y después se normaliza.
- BPE y WordPiece aprenden del corpus un vocabulario de subpalabras: las palabras frecuentes quedan enteras y las raras se arman con piezas, así que no quedan palabras fuera de vocabulario. El tamaño del vocabulario decide cuánto se corta.
- La lematización lleva cada palabra a su lema con ayuda de la gramática. El stemming corta terminaciones con reglas fijas: es más rápido y se equivoca más.

## Lecturas

- Jurafsky y Martin, *Speech and Language Processing*, 3.ª ed. (borrador), [capítulo 2: Words and Tokens](https://web.stanford.edu/~jurafsky/slp3/2.pdf).
- Hugging Face, *LLM Course*, capítulo 6: [BPE](https://huggingface.co/learn/llm-course/chapter6/5) y [WordPiece](https://huggingface.co/learn/llm-course/chapter6/6).
- Sennrich, Haddow y Birch (2016), [Neural Machine Translation of Rare Words with Subword Units](https://aclanthology.org/P16-1162/).
- spaCy, [Tokenization](https://spacy.io/usage/linguistic-features#tokenization).

## Ejercicios

### Ejercicio 1

Escriban `normalize_v2`, que conserve los números y los signos `¿?¡!` (útiles en análisis de sentimiento) pero siga pasando a minúsculas y quitando tildes. Pruébenla con «¡Subió 25% el dólar en 2023!».

<details><summary>Solución</summary>

```python
def normalize_v2(text: str, remove_tildes: bool = True) -> str:
    text = text.lower()
    # Conservamos dígitos y signos de exclamación e interrogación
    text = re.sub(r"[^a-z0-9ñüáéíóú¿?¡! ]", " ", text)
    if remove_tildes:
        for con_tilde, sin_tilde in zip("áéíóú", "aeiou"):
            text = text.replace(con_tilde, sin_tilde)
    return text


normalize_v2("¡Subió 25% el dólar en 2023!")
```

</details>

### Ejercicio 2

Extiendan el patrón del tokenizador para que reconozca menciones y hashtags (`@andres`, `#NLP`) como un solo token. ¿Dónde tiene que ir la alternativa nueva?

<details><summary>Solución</summary>

```python
# [@#]\w+ va antes de \w+; si no, \w+ se queda con "andres" y el @
# queda suelto
patron_redes = r'''(?x)
    [@#]\w+                 # menciones y hashtags
  | (?:[A-ZÁÉÍÓÚ]+\.)+
  | \$?\d+(?:\.\d+)?%?
  | \w+(?:-\w+)*
  | [¡¿!.,;?():]
'''
nltk.regexp_tokenize("@andres dice que #NLP cuesta $12.40 :)", patron_redes)
```

</details>

### Ejercicio 3

Elijan cuatro palabras de su área de trabajo y comparen el lema y el stem. ¿En cuáles el stemmer deja algo que no es una palabra? ¿En cuáles se equivoca el lematizador?

<details><summary>Solución</summary>

```python
mis_palabras = ["jugando", "jugadores", "juguetón", "abogados"]
for palabra in mis_palabras:
    lema = nlp(palabra)[0].lemma_
    print(f"{palabra:12s} lema: {lema:12s} stem: {stemmer.stem(palabra)}")
```

</details>

## Para explorar

Celdas para probar por su cuenta. En Colab, las dos con formulario se pueden cambiar sin tocar el código.

Normalicen cualquier noticia del corpus:

In [24]:
#@title Probar `normalize()` sobre un documento
indice = 0  #@param {type:"slider", min:0, max:4999, step:1}
remove_tildes = True  #@param {type:"boolean"}

texto = spanish_news["train"][indice]["text"]
print("Original:   ", texto)
print("Normalizado:", normalize(texto, remove_tildes=remove_tildes))

Original:    De no recibir más dinero, las raciones de la ONU en Siria se terminarán en dos meses. En un informe al Consejo de Seguridad de la ONU, Amos dijo que las raciones del PMA destinadas a los 4.000.000 de sirios ya han sido recortadas para poder llegar a la mayor cantidad de personas posible. Amos también hizo un llamado a juntar suministros para proteger a los sirios del frío, en miras al próximo invierno.
Normalizado: de no recibir mas dinero  las raciones de la onu en siria se terminaran en dos meses  en un informe al consejo de seguridad de la onu  amos dijo que las raciones del pma destinadas a los           de sirios ya han sido recortadas para poder llegar a la mayor cantidad de personas posible  amos tambien hizo un llamado a juntar suministros para proteger a los sirios del frio  en miras al proximo invierno 


Comparen `split()` y spaCy en frases difíciles:

In [25]:
casos_dificiles = [
    "El Sr. Pérez, ex-presidente, llegó a la cumbre a las 10:30 a.m.",
    "La ONU informó que el PIB creció un 2.5% durante el T3 de 2023.",
    "¿Habrá elecciones este año? ¡Todo indica que sí!",
    "El Banco Central subió la tasa hace 3-4 semanas, según la agencia.",
]

for texto in casos_dificiles:
    print("Texto:", texto)
    print("  split():", texto.split())
    print("  spaCy:  ", [token.text for token in spacy_tokenizer(texto)])
    print()

Texto: El Sr. Pérez, ex-presidente, llegó a la cumbre a las 10:30 a.m.
  split(): ['El', 'Sr.', 'Pérez,', 'ex-presidente,', 'llegó', 'a', 'la', 'cumbre', 'a', 'las', '10:30', 'a.m.']
  spaCy:   ['El', 'Sr.', 'Pérez', ',', 'ex-presidente', ',', 'llegó', 'a', 'la', 'cumbre', 'a', 'las', '10:30', 'a.m', '.']

Texto: La ONU informó que el PIB creció un 2.5% durante el T3 de 2023.
  split(): ['La', 'ONU', 'informó', 'que', 'el', 'PIB', 'creció', 'un', '2.5%', 'durante', 'el', 'T3', 'de', '2023.']
  spaCy:   ['La', 'ONU', 'informó', 'que', 'el', 'PIB', 'creció', 'un', '2.5%', 'durante', 'el', 'T3', 'de', '2023', '.']

Texto: ¿Habrá elecciones este año? ¡Todo indica que sí!
  split(): ['¿Habrá', 'elecciones', 'este', 'año?', '¡Todo', 'indica', 'que', 'sí!']
  spaCy:   ['¿', 'Habrá', 'elecciones', 'este', 'año', '?', '¡', 'Todo', 'indica', 'que', 'sí', '!']

Texto: El Banco Central subió la tasa hace 3-4 semanas, según la agencia.
  split(): ['El', 'Banco', 'Central', 'subió', 'la', 'tasa', 'h

Comparen el lema y el stem de cualquier palabra:

In [26]:
#@title Compara lema y stem de una palabra
palabra = "corriendo"  #@param {type:"string"}

print(f"Lema: {nlp(palabra)[0].lemma_}")
print(f"Stem: {stemmer.stem(palabra)}")

Lema: correr
Stem: corr
